In [11]:
# -----------------------------
# STEP 1: Import Libraries
# -----------------------------
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from tabulate import tabulate
import os

# -----------------------------
# STEP 2: Load Dataset
# -----------------------------
print("Current Directory:", os.getcwd())

df = pd.read_csv("Customer_support_data.csv")
print("Dataset loaded successfully.\n")

# -----------------------------
# STEP 3: Show Columns in Table Format
# -----------------------------
columns_df = pd.DataFrame(df.columns, columns=['Column Names'])
print("Columns in Dataset:")
print(tabulate(columns_df, headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 4: Show Sample Data in Table Format
# -----------------------------
data = df[['Unique id', 'Product_category', 'CSAT Score']]
print("Sample Data (first 5 rows):")
print(tabulate(data.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 5: Create User-Item Matrix
# -----------------------------
user_item_matrix = data.pivot_table(
    index='Unique id',
    columns='Product_category',
    values='CSAT Score'
).fillna(0)

print("User-Item Matrix Sample:")
print(tabulate(user_item_matrix.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 6: Normalize Ratings
# -----------------------------
scaler = MinMaxScaler()
normalized_matrix = scaler.fit_transform(user_item_matrix)

normalized_df = pd.DataFrame(
    normalized_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

print("Normalized User-Item Matrix Sample:")
print(tabulate(normalized_df.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 7: Compute User Similarity
# -----------------------------
user_similarity = cosine_similarity(normalized_df)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=normalized_df.index,
    columns=normalized_df.index
)

print("User Similarity Matrix Sample (first 5 users):")
print(tabulate(user_similarity_df.iloc[:5, :5], headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 8: Define Recommendation Function
# -----------------------------
def recommend_products(user_id, top_n=5):
    """
    Recommend top N product categories for a given user based on User-Based CF.
    """
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    weighted_scores = np.zeros(normalized_df.shape[1])
    for sim_user, similarity in similar_users.items():
        weighted_scores += similarity * normalized_df.loc[sim_user].values

    recommendations = pd.Series(
        weighted_scores,
        index=normalized_df.columns
    ).sort_values(ascending=False)

    already_rated = user_item_matrix.loc[user_id]
    recommendations = recommendations[already_rated == 0]

    return recommendations.head(top_n)

# -----------------------------
# STEP 9: Generate Recommendations for a Sample User
# -----------------------------
sample_user = user_item_matrix.index[0]
recommended_products = recommend_products(sample_user, top_n=10)

recommendation_df = pd.DataFrame({
    'Product Category': recommended_products.index,
    'Recommendation Score': recommended_products.values
})

print(f"Top 10 Recommended Product Categories for Customer {sample_user}:")
print(tabulate(recommendation_df, headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 10: Generate Recommendations for First 10 Users
# -----------------------------
all_recommendations = []

for user in user_item_matrix.index[:10]:  # first 10 users for readability
    recs = recommend_products(user, top_n=5)
    for prod, score in recs.items():
        all_recommendations.append([user, prod, score])

all_recommendations_df = pd.DataFrame(all_recommendations, columns=['Customer ID', 'Product Category', 'Recommendation Score'])

print("Top Recommendations for First 10 Users:")
print(tabulate(all_recommendations_df, headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 11: Summary Statistics
# -----------------------------
print("Total Customers:", user_item_matrix.shape[0])
print("Total Product Categories:", user_item_matrix.shape[1])
print("Recommendation System implemented successfully with professional tables.")

# -----------------------------
# STEP 12: Save Recommendations to CSV
# -----------------------------
output_path = 'week10_recommendations.csv'  # saved in the current folder (week 10)
all_recommendations_df.to_csv(output_path, index=False)
print(f"Week 10 recommendations saved successfully to: {output_path}")


Current Directory: f:\semester 7\DataScience_AI_Project\Week_10
Dataset loaded successfully.

Columns in Dataset:
╒════╤═════════════════════════╕
│    │ Column Names            │
╞════╪═════════════════════════╡
│  0 │ Unique id               │
├────┼─────────────────────────┤
│  1 │ channel_name            │
├────┼─────────────────────────┤
│  2 │ category                │
├────┼─────────────────────────┤
│  3 │ Sub-category            │
├────┼─────────────────────────┤
│  4 │ Customer Remarks        │
├────┼─────────────────────────┤
│  5 │ Order_id                │
├────┼─────────────────────────┤
│  6 │ order_date_time         │
├────┼─────────────────────────┤
│  7 │ Issue_reported at       │
├────┼─────────────────────────┤
│  8 │ issue_responded         │
├────┼─────────────────────────┤
│  9 │ Survey_response_Date    │
├────┼─────────────────────────┤
│ 10 │ Customer_City           │
├────┼─────────────────────────┤
│ 11 │ Product_category        │
├────┼──────────────────────